In [ ]:
import pandas as pd
import os
import numpy as np 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
main_df=pd.read_csv("main_strains_diameters_for_plotting_for_paper.csv") # main_strains_swarm_diameters_for_plotting_for_paper.csv

In [ ]:
main_df

In [ ]:
#removing gldM data from plotting
no_gldm_df=main_df[main_df['Strain']!="∆GldM"].copy(deep=True)

In [ ]:
# no_gldm_df
no_gldm_df.shape

# new figure for 2026 paper

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# using csv that also has gldm data ; this code ignores the gldm data
# no_gldm_df with columns: "Strain" (categorical) and "Diameter" (numeric)


# Panel / style (match other fig)

# sns.set(style="white", font_scale=1.2)

sns.set(style="darkgrid", font_scale=1.2)

fig, ay = plt.subplots(figsize=(3, 3), dpi=600)



ay.grid(False)                 # turn all off
ay.grid(axis="y", linestyle="-", linewidth=0.8)


for spine in ay.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1.0)

# Stable order of strains
order = list(no_gldm_df["Strain"].unique())

# Map each strain to a numeric x position (tight, like your reference)
positions = np.arange(len(order)) + 1.0  # 1,2,3,...

# Choose a snug geometry similar to your example
# (You can tweak width slightly without changing the "look")
width = 0.85

# Extract values per strain (list-of-arrays for matplotlib boxplot)
vals = [no_gldm_df.loc[no_gldm_df["Strain"] == s, "Diameter"].dropna().to_numpy()
        for s in order]

# -----------------------------
# Boxplot (matplotlib) — colored boxes, no caps
# -----------------------------
bp = ay.boxplot(
    vals,
    positions=positions,
    widths=width,
    patch_artist=True,
    showfliers=False,
    showcaps=False,  # remove horizontal caps
    whiskerprops=dict(linewidth=1.2, color="dimgray"),
    medianprops=dict(linewidth=1.4, color="dimgray"),
    boxprops=dict(linewidth=1.2, color="dimgray", facecolor="gainsboro"),
)

# # Color palette: repeat if you have >4 strains
# colors = ["#2C7BB6", "#FDAE61", "#D7191C", "#ABD9E9"]
# for i, patch in enumerate(bp["boxes"]):
#     patch.set_facecolor(colors[i % len(colors)])

# -----------------------------
# Overlay individual points (black, faint, slight jitter)
# -----------------------------
rng = np.random.default_rng(7)  # deterministic jitter
for x, y in zip(positions, vals):
    jitter = rng.normal(0, 0.04, size=len(y))  # same "feel" as your example
    ay.scatter(
        np.full(len(y), x) + jitter,
        y,
        s=33,
        color="goldenrod",
        alpha=0.6,
        zorder=3
    )

# -----------------------------
# Axes formatting (panel-like, snug)
# -----------------------------
ay.set_xlabel("Strain")
ay.set_ylabel("Diameter (mm)")

# If you want NO x tick labels (like the example):
# ay.set_xticks([])

# If you want strain labels but keep the same aesthetic:
ay.set_xticks(positions)
ay.set_xticklabels(order, rotation=0)

# Tight bounds with a little breathing room (similar to your 0.6..4.44 trick)
ay.set_xlim(positions[0] - 0.5, positions[-1] + 0.5)
 
# Keep your y-range choice automatic, or set explicitly if you want:
# ay.set_ylim(0, 1.6)

ay.margins(x=0.01)
plt.tight_layout(pad=0.25)

plt.savefig("diameter_boxpoints_panel2.png", dpi=600, bbox_inches="tight")
plt.show()

stat

In [ ]:
# --- Statistical analysis for swarming diameters ---
# 1. Technical replicates averaged within each biological replicate (Exp1/2/3)
# 2. Randomized block ANOVA with experimental day as block
# 3. Tukey HSD post-hoc: all pairwise comparisons

import pandas as pd
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Average technical replicates within each biological replicate
bio_means = no_gldm_df.groupby(['Strain', 'Repeat'])['Diameter'].mean().reset_index()

# Randomized block ANOVA
model = ols('Diameter ~ C(Strain) + C(Repeat)', data=bio_means).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("--- ANOVA (Swarming Diameter) ---")
print(anova_table)

# Tukey HSD post-hoc (all pairwise)
posthoc = pairwise_tukeyhsd(endog=bio_means['Diameter'], groups=bio_means['Strain'], alpha=0.05)
print("\n--- Tukey HSD (all pairwise) ---")
print(posthoc)